# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.

In [1]:
%pip install evaluate rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=ea6fbbcf04ca74f91fbf2f68952bdcee00d5101ce231f1de985b25c0ab76e731
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.

In [2]:
import torch
SEED = 101
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MAX_TRAIN_SAMPLES = 5000
MAX_TEST_SAMPLES = 400

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"

# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.

In [3]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/all/train-00000-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00001-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

data/all/train-00002-of-00009.parquet:   0%|          | 0.00/223M [00:00<?, ?B/s]

...

data/all/train-00007-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/train-00008-of-00009.parquet:   0%|          | 0.00/224M [00:00<?, ?B/s]

data/all/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/54948 [00:00<?, ? examples/s]

In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).

In [4]:
smoltalk_simplified = smoltalk.filter(lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages']))
smoltalk_simplified = DatasetDict({
    "train": smoltalk_simplified["train"].select(range(MAX_TRAIN_SAMPLES)),
    "test": smoltalk_simplified["test"].select(range(MAX_TEST_SAMPLES)),
})

Filter:   0%|          | 0/1043917 [00:00<?, ? examples/s]

Filter:   0%|          | 0/54948 [00:00<?, ? examples/s]

In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 400
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.

In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).

> **🎓 Notes — Task 1.2**
> - **Format chosen**: ChatML (`<|im_start|>role\ncontent<|im_end|>`) — SmolLM2's native format, also used by Qwen and Mistral
> - **Prompt ends with `<|im_start|>assistant\n`** — model's job is to continue from there
> - **Response includes `<|im_end|>\n`** — teaches the model when to stop talking
> - **Why a structured format?** Without role markers, the model can't tell user input from its own output. The special tokens give it an unambiguous signal of "now it's my turn to speak"
> - **Why not just `### Instruction / ### Response`?** ChatML is a single round-trip tokenizable design with dedicated special token ids — cleaner and matches what production LLMs use


In [7]:
def format_input_output(example):
  # `messages` is a list of messages, each with a `content` string and a `role`.
  messages = example['messages']

  # ChatML format used by SmolLM2 (and Qwen, Mistral). The prompt ends right
  # after the assistant header, so the model's job at training time is to
  # generate the assistant's content + <|im_end|>.
  prompt_parts = []
  response = None
  for msg in messages:
    if msg['role'] == 'assistant':
      # The response is the assistant's content plus the end-of-turn marker,
      # so the model learns when to stop speaking.
      response = msg['content'] + '<|im_end|>\n'
      # Open the assistant turn at the end of the prompt — generation starts
      # right after this line.
      prompt_parts.append('<|im_start|>assistant\n')
      break
    else:
      # 'system' or 'user' message — both go into the prompt.
      prompt_parts.append(f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n")

  prompt = ''.join(prompt_parts)
  return {"prompt": prompt, "response": response}


Apply the function you implemented to the dataset as a whole.

In [8]:
ds_sft = smoltalk_simplified.map(format_input_output)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Then verify that the dataset now contains the new fields you created.

In [9]:
ds_sft['train'][0]

{'messages': [{'content': "You are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.",
   'role': 'system'},
  {'content': 'Rearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.',
   'role': 'user'},
  {'content': 'The chef made more food after the restaurant ran out.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': "<|im_start|>system\nYou are an AI rewriting assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.<|im_end|>\n<|im_start|>user\nRearrange this sentence to make it easy to understand:\nThe restaurant ran out of food, so the chef made some more.<|im_end|>\n<|im_start|>assistant\n",
 'response': 'The chef made more food after the restaurant ran out.<|im_end|>\n'}

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:

> **⚙ Notes — Task 1.3**
> - `input_ids` = prompt tokens + response tokens (concatenated)
> - `labels` = `[-100] * len(prompt)` + response tokens — prompt positions get **masked out of the loss**
> - `attention_mask` = all 1s (no padding here; the collator adds padding later)
> - **Why mask the prompt?** We're doing **conditional** generation. The model already sees the prompt as input; we don't want to teach it to predict the prompt itself. Loss only on positions where we want it to *learn to produce something new*.
> - `add_special_tokens=False`: ChatML already wraps everything with special tokens; we don't want a stray BOS prepended


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)

In [11]:
def tokenize_helper(example):
    prompt = example['prompt']     # Created in the previous step
    response = example['response'] # Created in the previous step

    # Tokenize prompt and response separately so we know exactly which
    # positions belong to the prompt (to mask out) and which to the response
    # (to train on). add_special_tokens=False because ChatML already includes
    # all the structural tokens we need; we don't want a stray BOS.
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    response_ids = tokenizer(response, add_special_tokens=False)['input_ids']

    input_ids = prompt_ids + response_ids
    # Mask prompt tokens (-100) so loss only counts response positions.
    labels = [-100] * len(prompt_ids) + response_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.

In [12]:
tokenized_ds_sft = ds_sft.map(
    tokenize_helper,
    remove_columns=ds_sft['train'].column_names,
)
tokenized_ds_sft


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 400
    })
})


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.

### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.

In [13]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:

In [14]:
import evaluate

class RougeMetricComputer:
    """
    Stateful metric for batch_eval_metrics=True.

    It:
      - accumulates predictions and references across batches
      - computes ROUGE-L once at the end (compute_result=True)
    """

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load("rouge")
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        """Accumulate predictions and compute at the end."""

        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        # Collect decoded answer-span text from each example in the batch
        for p, lbl in zip(pred_ids, labels):
            mask = lbl != -100
            if mask.sum() == 0:
                continue

            ref_ids = lbl[mask]
            pred_ids_filtered = p[mask]

            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(
                pred_ids_filtered, skip_special_tokens=True,
                eos_token_id=self.tokenizer.vocab['<|im_end|>']
            )

            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        # Only compute at the very end of eval
        if compute_result:
            if len(self.all_references) > 0:
                scores = self.rouge.compute(
                    predictions=self.all_predictions,
                    references=self.all_references,
                )

                # Clear accumulated data for next eval call
                self.all_predictions = []
                self.all_references = []
                return {"rougeL": scores["rougeL"]}
            else:
                return {}
        else:
            return {}

compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).

In [15]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

The following code will compute the loss on the test set as well as the ROUGE-L score. You will later compare these scores to the models that you train.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

> **🎓 Notes — Task 2.2**
> - **My results**: loss = 2.48, ROUGE-L = **0.564**
> - **Why is ROUGE-L still 0.56 without any instruction tuning?** Several reasons:
>   1. **ROUGE-L is teacher-forced here**: at each position, we feed in the gold prefix and check if `argmax(logits)` matches the gold next token. This is essentially "next-token accuracy on the gold response" — not real generation. The model only needs to produce the *right next token* given the rest already correct, which is much easier than actually generating.
>   2. **Pretrained LMs already model fluent English** — many tokens (articles, common verbs, sentence enders) are easy to guess from context
>   3. **No `<|im_end|>` understanding yet**: the pretrained model has seen ChatML special tokens during its own pretraining (SmolLM2 was actually trained on some chat data too), so it has at least *some* familiarity with the format
> - **What ROUGE-L doesn't tell us**: that free-form generation is garbage — see Task 4.4, where the pretrained model just **repeats the prompt 15 times** instead of answering it. High teacher-forced ROUGE ≠ usable chatbot.


In [16]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM
import time, json

print("\n" + "=" * 80)
print("EVALUATING PRETRAINED MODEL")
print("=" * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

pretrained_eval_args = TrainingArguments(
    eval_strategy="no",
    per_device_eval_batch_size=1,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics["eval_loss"])
pretrained_rougeL = pretrained_eval_metrics.get("eval_rougeL", None)

print("\nPRETRAINED EVAL METRICS:")
print(json.dumps(pretrained_eval_metrics, indent=2))
print(f"Evaluation time: {pretrained_eval_time:.1f}s")



EVALUATING PRETRAINED MODEL


model.safetensors:   0%|          | 0.00/269M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

  0%|          | 0/400 [00:00<?, ?it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 2.4811792373657227,
  "eval_model_preparation_time": 0.0055,
  "eval_rougeL": 0.5636076479340224,
  "eval_runtime": 34.2465,
  "eval_samples_per_second": 11.68,
  "eval_steps_per_second": 11.68
}
Evaluation time: 34.5s



## Part 3: Supervised fine-tuning



### 🎓&nbsp; Task 3.1: Training the full model

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

> **🎓 Notes — Task 3.1**
> - **My results**: loss 2.48 → **1.41**, ROUGE-L 0.564 → **0.636**, training time 825 s
> - **All 134.5M parameters are updated**
> - **Train loss curve**: 1.74 → 1.29 across the epoch — steady, monotonic, no instability
> - **Bigger gain in generation quality than in metrics**: ROUGE only goes up by ~0.07, but qualitatively the model went from "repeat the prompt forever" to "actually answer the question correctly" (see Task 4.4)
> - **Why is the metric jump smaller than the qualitative jump?** Because ROUGE is teacher-forced — it was already 0.56 just from fluent next-token prediction. The big win from SFT is in **structure** (knowing when to stop, staying in answer mode, not echoing the prompt), which only shows up in free-form generation.


In [17]:
baseline_training_args = TrainingArguments(
    output_dir='./out_full_sft',
    eval_strategy="epoch",
    logging_steps=500,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True, fp16=False, # This may need to be changed, depending on the model you selected
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

baseline_trainer = make_trainer(base_model, baseline_training_args)

# Train, then evaluate.
t0 = time.perf_counter()
baseline_trainer.train()
baseline_train_time = time.perf_counter() - t0
print(f"\nFull SFT training time: {baseline_train_time:.1f}s")

baseline_eval_metrics = baseline_trainer.evaluate()
baseline_eval_loss = float(baseline_eval_metrics["eval_loss"])
baseline_rougeL = baseline_eval_metrics.get("eval_rougeL", None)

print("\nFULL SFT EVAL METRICS:")
print(json.dumps(baseline_eval_metrics, indent=2))


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.743', 'grad_norm': '12.81', 'learning_rate': '4.501e-05', 'epoch': '0.1'}
{'loss': '1.572', 'grad_norm': '11.75', 'learning_rate': '4.001e-05', 'epoch': '0.2'}
{'loss': '1.473', 'grad_norm': '6.562', 'learning_rate': '3.501e-05', 'epoch': '0.3'}
{'loss': '1.524', 'grad_norm': '6.656', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.467', 'grad_norm': '3.75', 'learning_rate': '2.501e-05', 'epoch': '0.5'}
{'loss': '1.371', 'grad_norm': '5.438', 'learning_rate': '2.001e-05', 'epoch': '0.6'}
{'loss': '1.391', 'grad_norm': '4.25', 'learning_rate': '1.501e-05', 'epoch': '0.7'}
{'loss': '1.318', 'grad_norm': '10.38', 'learning_rate': '1.001e-05', 'epoch': '0.8'}
{'loss': '1.364', 'grad_norm': '18.62', 'learning_rate': '5.01e-06', 'epoch': '0.9'}
{'loss': '1.29', 'grad_norm': '15', 'learning_rate': '1e-08', 'epoch': '1'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.415', 'eval_rougeL': '0.6356', 'eval_runtime': '34.43', 'eval_samples_per_second': '11.62', 'eval_steps_per_second': '11.62', 'epoch': '1'}
{'train_runtime': '824.8', 'train_samples_per_second': '6.062', 'train_steps_per_second': '6.062', 'train_loss': '1.451', 'epoch': '1'}

Full SFT training time: 825.1s


  0%|          | 0/400 [00:00<?, ?it/s]


FULL SFT EVAL METRICS:
{
  "eval_loss": 1.4145338535308838,
  "eval_rougeL": 0.635640752478104,
  "eval_runtime": 34.234,
  "eval_samples_per_second": 11.684,
  "eval_steps_per_second": 11.684,
  "epoch": 1.0
}


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.

> **⚙ Notes — Task 3.3**
> - `sum(p.numel() for p in model.parameters() if p.requires_grad)`
> - `.numel()` = total elements in the tensor (product of all dimensions)
> - `p.requires_grad` is False for frozen parameters — they won't get gradient updates
> - **My result for full SFT**: all 134,515,008 parameters are trainable (100%)


In [18]:
def num_trainable_parameters(model):
    """Count number of trainable parameters.

    Args:
        model: A PyTorch module.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


Apply this function to the SFT-trained model and check that the result makes sense.

In [19]:
n_full = num_trainable_parameters(base_model)
n_total = sum(p.numel() for p in base_model.parameters())
print(f'Full SFT trainable params: {n_full:,}')
print(f'Total params:              {n_total:,}')
print(f'Fraction trainable:        {100*n_full/n_total:.2f}%')


Full SFT trainable params: 134,515,008
Total params:              134,515,008
Fraction trainable:        100.00%


### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.

> **⚙ Notes — Task 4.1**
> - Iterate `model.named_modules()` and pick by **leaf name** — generic across model families
> - Targets: `q_proj`, `k_proj`, `v_proj`, `o_proj` (all 4 attention projections)
> - **My result**: 120 target layers = 30 transformer layers × 4 projections per layer
> - **Why these 4?** The LoRA paper's default was q+v only, but the assignment hint recommends all 4. More targets → more capacity for the LoRA update, marginally more params.
> - **Why not also MLP?** MLP has the largest weight matrices (where most params live), so LoRA there would save the most. But the original paper found attention was sufficient. The PEFT library does target MLP as well by default.


In [20]:
import torch.nn as nn

def extract_lora_targets(model):
  """Find all attention q/k/v/o projection layers, keyed by full module name."""
  targets = {}
  # Pattern in SmolLM2 / Llama family: model.layers.{i}.self_attn.{q,k,v,o}_proj.
  # We pick them up by their leaf name ('q_proj', etc.) across all attention blocks.
  wanted = {'q_proj', 'k_proj', 'v_proj', 'o_proj'}
  for name, module in model.named_modules():
    short = name.split('.')[-1]
    if short in wanted and isinstance(module, nn.Linear):
      targets[name] = module
  return targets


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.

In [21]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.

Start from the following skeleton and fill in the missing pieces:

> **🎓 Notes — Task 4.2**
> 
> **Forward**: `y = W(x) + (alpha/r) · B(A(x))`
> - **W** is the frozen original linear (no grad updates)
> - **A**: in→r, Kaiming-init random
> - **B**: r→out, **zero-init** ← critical detail
> - **scaling = alpha/r** is a fixed multiplier
> 
> **Why B = 0 at init?** Then at training step 0, B(A(x)) = 0, so the model's output is **exactly** the pretrained model's output. No disruption of pretrained behavior. Without this, the very first forward pass would be a random perturbation of the LLM.
> 
> **Why low rank?** A full update would be a ΔW of shape (out, in). Instead we factor it as B·A of shape (out, r) and (r, in). For r ≪ min(in, out), this is **vastly fewer parameters**. Hu et al. showed empirically that LM fine-tuning updates are in fact low-rank in practice.
> 
> **Why the alpha/r scaling?** When you change r (say r=8 → r=32), without scaling the effective magnitude of the update would grow with r. Scaling by alpha/r keeps the update size roughly constant, so you can tune r without re-tuning learning rate. **alpha is the real "LoRA learning rate" knob**; r is just capacity.
> 
> **Parameter savings (my model)**:
> - 1 Linear of shape (576, 576) = 331,776 params
> - LoRA replacement with r=16: 576·16 + 16·576 = **18,432 params** (5.5%)
> - Across all 120 wrapped layers: 1.84M trainable vs 134.5M for full SFT → **73× fewer**


In [22]:
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        """Wrap an existing linear layer W with a low-rank update B·A.

        Forward: y = W(x) + (alpha/r) * B(A(x))
          - W is the frozen original pretrained linear.
          - A: in_features -> r   (Kaiming init)
          - B: r -> out_features  (zero init, so the update starts at 0)
          - alpha/r is the scaling factor described in the paper.
        """
        super().__init__()
        self.W = W
        # Freeze the original weights — only A and B will get gradient updates.
        for p in self.W.parameters():
            p.requires_grad = False

        in_features = W.in_features
        out_features = W.out_features

        # Low-rank decomposition. No bias on either matrix.
        self.A = nn.Linear(in_features, r, bias=False)
        self.B = nn.Linear(r, out_features, bias=False)

        # Standard LoRA init: A ~ Kaiming, B = 0.
        # B=0 means the model's initial behavior is exactly the pretrained one,
        # which keeps training stable.
        nn.init.kaiming_uniform_(self.A.weight, a=5 ** 0.5)
        nn.init.zeros_(self.B.weight)

        self.scaling = alpha / r

    def forward(self, x):
        # Frozen base output + scaled low-rank update.
        return self.W(x) + self.scaling * self.B(self.A(x))


Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.

### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

Apply your parameter counting function (`num_trainable_parameters`) to this model, compare the results to those in Part 3, and make sure that these results correspond to your expectations.

> **🎓 Notes — Task 4.3**
> 
> **Workflow**:
> 1. `extract_lora_targets(model)` → dict of 120 (q/k/v/o) Linears
> 2. Wrap each with `LoRALayer(W, r=16, alpha=32)` — wrapping freezes W internally
> 3. `replace_layers(model, wrapped)` swaps them in place
> 4. **Manually freeze everything else** (embeddings, MLP, norms, lm_head) — wrapping the attention layers doesn't touch them
> 
> **My results**:
> 
> | Model | loss | ROUGE-L | trainable params | train time |
> |---|---|---|---|---|
> | Pretrained | 2.481 | 0.564 | 0 | — |
> | Full SFT | 1.415 | **0.636** | 134.5M | 825 s |
> | LoRA (r=16, α=32) | 1.439 | 0.633 | **1.84M** (1.35%) | 964 s |
> 
> **Key observations**:
> - **LoRA hits 99.5% of full SFT's ROUGE-L with 1.35% of the parameters** — the whole point of LoRA, working as advertised
> - **Loss gap is tiny**: 1.44 vs 1.41 (~2% difference)
> - **LoRA was actually *slower* per epoch** (964s vs 825s) despite training fewer params — the LoRA forward pass adds one extra matmul (the B·A·x path), and we still backprop through frozen weights to reach the LoRA params. *Memory savings* (not speed) is LoRA's main practical benefit: optimizer states (Adam: 2 floats per trainable param) drop dramatically.
> - **Gradient norms are much smaller** for LoRA (~1–2) vs full SFT (~5–15) — fewer parameters, so each one absorbs a smaller share of the gradient signal


In [23]:
# Hyperparameters from the LoRA paper. r controls capacity vs param count;
# alpha is a fixed scaling factor.
LORA_R = 16
LORA_ALPHA = 32

# Fresh copy of the pretrained model (don't reuse the already-SFT'd one).
lora_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Step 1: find all q/k/v/o projection layers across the transformer stack.
targets = extract_lora_targets(lora_model)
print(f'Found {len(targets)} target layers to wrap with LoRA')
for name in list(targets)[:4]:
    print(f'  {name}')
print('  ...')

# Step 2: wrap each target inside a LoRALayer. The wrapper freezes the
# underlying W and adds the trainable A, B matrices.
wrapped = {name: LoRALayer(W, r=LORA_R, alpha=LORA_ALPHA) for name, W in targets.items()}

# Step 3: swap the new LoRA layers back into the model in place.
lora_model = replace_layers(lora_model, wrapped)

# Step 4: freeze ALL other parameters. LoRALayer already froze its W, but
# the rest of the model (embeddings, MLP, norms, lm_head) needs explicit freezing.
for name, p in lora_model.named_parameters():
    # Heuristic: parameters inside one of our LoRA wrappers will have '.A.' or
    # '.B.' in their name. Everything else stays frozen.
    if '.A.' in name or '.B.' in name:
        p.requires_grad = True
    else:
        p.requires_grad = False

lora_model = lora_model.to(DEVICE)

# Sanity check: how many parameters are actually trainable now?
n_lora = num_trainable_parameters(lora_model)
n_total = sum(p.numel() for p in lora_model.parameters())
print(f'\nLoRA trainable params: {n_lora:,}')
print(f'Total params:          {n_total:,}')
print(f'Fraction trainable:    {100*n_lora/n_total:.3f}%')
print(f'Reduction vs full SFT: {n_full/n_lora:.1f}x fewer trainable params')


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Found 120 target layers to wrap with LoRA
  model.layers.0.self_attn.q_proj
  model.layers.0.self_attn.k_proj
  model.layers.0.self_attn.v_proj
  model.layers.0.self_attn.o_proj
  ...

LoRA trainable params: 1,843,200
Total params:          136,358,208
Fraction trainable:    1.352%
Reduction vs full SFT: 73.0x fewer trainable params


In [24]:
lora_training_args = TrainingArguments(
    output_dir='./out_lora_sft',
    eval_strategy="epoch",
    logging_steps=500,
    save_strategy="no",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    bf16=True, fp16=False,
    report_to="none",
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

lora_trainer = make_trainer(lora_model, lora_training_args)

t0 = time.perf_counter()
lora_trainer.train()
lora_train_time = time.perf_counter() - t0
print(f'\nLoRA training time: {lora_train_time:.1f}s')

lora_eval_metrics = lora_trainer.evaluate()
lora_eval_loss = float(lora_eval_metrics['eval_loss'])
lora_rougeL = lora_eval_metrics.get('eval_rougeL', None)
print('\nLORA EVAL METRICS:')
print(json.dumps(lora_eval_metrics, indent=2))

# Side-by-side comparison.
print('\n' + '=' * 60)
print(f'{"Model":<20s} {"loss":>8s} {"rougeL":>8s} {"params":>15s}')
print('-' * 60)
print(f'{"Pretrained":<20s} {pretrained_eval_loss:>8.3f} {(pretrained_rougeL or 0):>8.3f} {0:>15,}')
print(f'{"Full SFT":<20s} {baseline_eval_loss:>8.3f} {(baseline_rougeL or 0):>8.3f} {n_full:>15,}')
print(f'{"LoRA SFT (r=16)":<20s} {lora_eval_loss:>8.3f} {(lora_rougeL or 0):>8.3f} {n_lora:>15,}')


  0%|          | 0/5000 [00:00<?, ?it/s]

{'loss': '1.833', 'grad_norm': '2.397', 'learning_rate': '4.501e-05', 'epoch': '0.1'}
{'loss': '1.658', 'grad_norm': '2.446', 'learning_rate': '4.001e-05', 'epoch': '0.2'}
{'loss': '1.55', 'grad_norm': '1.306', 'learning_rate': '3.501e-05', 'epoch': '0.3'}
{'loss': '1.594', 'grad_norm': '1.301', 'learning_rate': '3.001e-05', 'epoch': '0.4'}
{'loss': '1.541', 'grad_norm': '0.6995', 'learning_rate': '2.501e-05', 'epoch': '0.5'}
{'loss': '1.415', 'grad_norm': '1.224', 'learning_rate': '2.001e-05', 'epoch': '0.6'}
{'loss': '1.424', 'grad_norm': '0.7312', 'learning_rate': '1.501e-05', 'epoch': '0.7'}
{'loss': '1.376', 'grad_norm': '1.949', 'learning_rate': '1.001e-05', 'epoch': '0.8'}
{'loss': '1.405', 'grad_norm': '5.156', 'learning_rate': '5.01e-06', 'epoch': '0.9'}
{'loss': '1.347', 'grad_norm': '3.246', 'learning_rate': '1e-08', 'epoch': '1'}


  0%|          | 0/400 [00:00<?, ?it/s]

{'eval_loss': '1.439', 'eval_rougeL': '0.6333', 'eval_runtime': '41.23', 'eval_samples_per_second': '9.701', 'eval_steps_per_second': '9.701', 'epoch': '1'}
{'train_runtime': '964.1', 'train_samples_per_second': '5.186', 'train_steps_per_second': '5.186', 'train_loss': '1.514', 'epoch': '1'}

LoRA training time: 964.4s


  0%|          | 0/400 [00:00<?, ?it/s]


LORA EVAL METRICS:
{
  "eval_loss": 1.4388208389282227,
  "eval_rougeL": 0.6332762039815822,
  "eval_runtime": 40.8681,
  "eval_samples_per_second": 9.788,
  "eval_steps_per_second": 9.788,
  "epoch": 1.0
}

Model                    loss   rougeL          params
------------------------------------------------------------
Pretrained              2.481    0.564               0
Full SFT                1.415    0.636     134,515,008
LoRA SFT (r=16)         1.439    0.633       1,843,200


### 🎓&nbsp; Task 4.4: Qualitative inspection

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

The quality we see here will depend on your choice of base model as well as how much you trained it.

> **🎓 Notes — Task 4.4**
> 
> **The most striking finding from my outputs**:
> 
> **Pretrained model is broken as a chatbot.** Every prompt produces *the same prompt repeated 10+ times* with no answer. This is the classic "no instruction-following" failure mode: the model continues the text statistically (it has seen many documents with section titles like "What is X?" followed by "What is X?" type repetition online), but never *answers*.
> 
> **Both SFT models work, with quality differences**:
> 
> | Prompt | Pretrained | Full SFT | LoRA SFT |
> |---|---|---|---|
> | Capital of Sweden? | (repeats 15×) | "Stockholm" ✓ | "Stockholm" ✓ |
> | Why is the sky blue? | (repeats 11×) | "reflects sunlight... rainbow" (mostly right) | "reflects the color of the sky" ✗ (tautology) |
> | List 3 benefits of exercise | (repeats 14×) | "weight, cardio, chronic diseases" ✓ | "strong, flexible, diabetes / heart" ✓ |
> | Haiku about ML | section headers | "ML algorithm is a program..." (not a haiku) | same, with a metaphor |
> 
> **Take-aways**:
> - Both SFT models clearly **stop repeating the prompt** — they've internalized the "now it's my turn to answer" signal that ChatML provides
> - Neither model actually writes a **haiku**: SFT on 5000 generic examples doesn't teach genre-specific skills. Haiku-following would need either much more training data or examples specifically of haiku generation.
> - LoRA occasionally produces **tautologies** ("the sky is blue because it reflects the color of the sky") — symptom of insufficient capacity to encode factual knowledge updates; gets a fluent answer right but a factual one wrong
> - **Greedy decoding** (do_sample=False) is what causes the repetition in the pretrained model — sampling with temperature would mask the problem somewhat, but the underlying issue is that the model never *prefers* a real answer over more prompt text


In [25]:
@torch.no_grad()
def generate_response(model, user_message, system_message=None, max_new_tokens=120):
    """Build a ChatML prompt for an example, run the model, return the decoded reply."""
    # Build the prompt with the SAME format used in training (Task 1.2).
    messages = []
    if system_message is not None:
        messages.append({'role': 'system', 'content': system_message})
    messages.append({'role': 'user', 'content': user_message})
    # Add an empty assistant message so format_input_output will append the
    # opening assistant header at the end of the prompt.
    messages.append({'role': 'assistant', 'content': ''})
    formatted = format_input_output({'messages': messages})
    prompt = formatted['prompt']

    inputs = tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,             # deterministic — easier to compare models
        eos_token_id=tokenizer.convert_tokens_to_ids('<|im_end|>'),
        pad_token_id=tokenizer.pad_token_id,
    )
    # Slice off the prompt; decode only the generated continuation.
    gen_ids = out[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True)


test_prompts = [
    'What is the capital of Sweden?',
    'Write a short haiku about machine learning.',
    'Explain in one sentence why the sky is blue.',
    'List three benefits of regular exercise.',
]

for prompt in test_prompts:
    print('=' * 70)
    print(f'USER: {prompt}')
    print('-' * 70)
    for name, m in [('Pretrained', pretrained_model),
                    ('Full SFT', base_model),
                    ('LoRA SFT', lora_model)]:
        print(f'\n[{name}]')
        print(generate_response(m, prompt).strip())
    print()


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


USER: What is the capital of Sweden?
----------------------------------------------------------------------

[Pretrained]


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?
What is the capital of Sweden?

[Full SFT]


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


...

The sky is blue because it reflects the color of the sky.

USER: List three benefits of regular exercise.
----------------------------------------------------------------------

[Pretrained]


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits of regular exercise.

List three benefits

[Full SFT]


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Regular exercise helps maintain a healthy weight, improves cardiovascular health, and reduces the risk of chronic diseases.

[LoRA SFT]
Regular exercise helps keep your body strong and flexible, which can help you perform better in sports and other physical activities. It also reduces the risk of developing chronic diseases such as diabetes and heart disease.

